# 00 — Environment Setup & GPU Check

Run this notebook first after creating your OpenShift AI Workbench.
It verifies GPU access, installs dependencies, and runs a smoke test.

**OpenShift AI Workbench settings:**
- Image: PyTorch (CUDA)
- Container size: Large (8+ CPU, 32+ GB RAM)
- Accelerator: NVIDIA GPU x1
- Persistent storage: >= 30 GB

## 1. GPU Verification

In [ ]:
!nvidia-smi

In [ ]:
!pip show torch

# if torch is not already provided:
#!pip install torch==2.10.0

In [ ]:
!pip show flash_attn

# if flash_attn is not already provided:
#!pip install flash_attn==2.8.3

In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU:             {gpu_name}")
    print(f"VRAM:            {vram_gb:.1f} GB")

    if vram_gb < 20:
        print("\n⚠  VRAM < 20 GB — Qwen3-8B in fp16 may not fit.")
        print("   The experiment notebooks will use 4-bit quantization (BitsAndBytesConfig).")
    else:
        print("\n✓  Enough VRAM for Qwen3-8B in fp16.")
else:
    raise RuntimeError(
        "No CUDA GPU detected. Make sure your Workbench has a GPU accelerator assigned."
    )

## 2. Environment & Dependencies

Set `HF_HOME` to persistent storage so downloads survive pod restarts.
Re-run the cell below if the pod was recreated. Pip will skip already-installed packages.

In [ ]:
import os
os.environ["HF_HOME"] = "/opt/app-root/src/.cache/huggingface"

!pip install git+https://github.com/NVIDIA/kvpress.git@v0.5.4 --no-deps
!pip install vllm==0.18.0 transformers==4.57.6 datasets==3.6.0 \
    rouge-score==0.1.2 matplotlib==3.10.8 pandas==2.3.3 \
    bitsandbytes==0.49.2 accelerate==1.12.0 fire==0.7.1

# kvpress evaluation framework dependencies (uv sync --extra eval)
!pip install rouge==1.0.1 jieba==0.42.1 fuzzywuzzy==0.18.0 bert-score==0.3.13 \
    --index-url https://pypi.org/simple/

## 3. Verify Imports

In [ ]:
from importlib.metadata import version
import kvpress, vllm

for pkg in ["kvpress", "vllm", "transformers", "datasets", "rouge-score", "matplotlib", "pandas"]:
    print(f"{pkg:20s} {version(pkg)}")

## 4. Verify kvpress Press Types

In [ ]:
from kvpress import KeyDiffPress, BlockPress, PrefillDecodingPress, CompressionRatioDecodingPress

press = PrefillDecodingPress(
    prefilling_press=BlockPress(press=KeyDiffPress(compression_ratio=0.5), block_size=128),
    decoding_press=CompressionRatioDecodingPress(
        base_press=KeyDiffPress(), target_compression_ratio=0.5,
    ),
)

print(f"PrefillDecodingPress: {press}")
print(f"  prefilling: {press.prefilling_press}")
print(f"  decoding:   {press.decoding_press}")
print("\n✓  Press types instantiated successfully.")

## 5. Smoke Test — CUDA Tensor Round-Trip

In [ ]:
x = torch.randn(1024, 1024, device="cuda", dtype=torch.float16)
y = x @ x.T
print(f"Matrix multiply on GPU OK — result shape: {y.shape}, dtype: {y.dtype}")
del x, y
torch.cuda.empty_cache()

## 6. Dataset Access Check

In [ ]:
from datasets import load_dataset

ds = load_dataset("alessiodevoto/paul_graham_essays", split="test")
print(f"Paul Graham essays loaded: {len(ds)} rows")
print(f"Columns: {ds.column_names}")
print(f"Context length: {len(ds[0]['context'].split())} words")
print(f"Needle: {ds[0]['needle'][:80]}...")
print(f"Question: {ds[0]['question']}")

## Done

If all cells above ran without errors, your environment is ready.  
Proceed to:
- `01_kvpress_fork_setup.ipynb` — install kvpress from fork (if needed)
- `02_kvpress_niah.ipynb` — KeyDiffPress NIAH experiments
- `03_vllm_fork_setup.ipynb` — install vLLM from fork (if needed)
- `04_vllm_niah.ipynb` — vLLM baseline NIAH experiments